In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from ALLCools.mcds import MCDS
import ALLCools
import xarray as xr

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

import glob

import os

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/'

In [ ]:
clust_path = f'/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clust_path, sep='\t', index_col=0)
# clusters.index = [f'{x}_indexed' for x in clusters.index]
clusters.shape

### standardize to same cells in clusters

In [ ]:
# obs_dim = 'cell'
# var_dim = 'chrom100k'

# indir = f'{projdir}/mcds/batch/{var_dim}'
# batches = np.array([str(i).zfill(3) for i in range(261)])
# mask = np.array([os.path.exists(f'{indir}/{x}_tmp') for x in batches])
# final_batches = batches[~mask]
# final_batches.shape

In [ ]:
n = 223
final_batches = [str(i).zfill(3) for i in range(223)]

In [ ]:
%%time

obs_dim = 'cell'
var_dim = 'chrom100k'

def load_mcds(batch):
    mcds_path = f'{projdir}/mcds/batch/{var_dim}/{batch}'
    mcds = MCDS.open(mcds_path, obs_dim = obs_dim,  
            var_dim = var_dim, engine = 'zarr')
    return mcds

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(load_mcds, final_batches), total=len(final_batches)))

In [ ]:
results

In [ ]:
mcds_merged = MCDS(xr.concat(results, dim=obs_dim))
mcds_merged.obs_dim = obs_dim
mcds_merged.var_dim = var_dim

In [ ]:
mcds_merged

In [ ]:
mask = pd.Index(mcds_merged['cell'].values).isin(clusters.index)
final_cells = mcds_merged['cell'].values[mask]
final_cells.shape

In [ ]:
mcds_merged.add_feature_cov_mean(var_dim=var_dim)

In [ ]:
mcds_merged.coords[f'{var_dim}_cov_mean'].values.shape, \
mcds_merged.coords[f'{var_dim}_cov_mean'].values.mean()

In [ ]:
mcg_pattern = 'CGN'

In [ ]:
mcg_pattern = 'CGN'
mcds_merged.add_mc_frac(
    normalize_per_cell=True,  # after calculating mC frac, per cell normalize the matrix
        clip_norm_value=10  # clip outlier values above 10 to 10
)

adata = mcds_merged.get_adata(mc_type=mcg_pattern,
                   var_dim=var_dim,
                   select_hvf=False)
adata.write_h5ad(f'{projdir}/h5ad/{var_dim}/raw_norm.h5ad')

### let's repeat for every other feature type

In [ ]:
var_dims = ['chrom100k', 'geneslop2k', 'chrom5k', 'peaks', 'dmrs']

In [ ]:
%%time

# for var_dim in var_dims:
for var_dim in var_dims[1:]:
    print(var_dim)
    ###
#     indir = f'{projdir}/mcds/batch/{var_dim}'
#     batches = np.array([str(i).zfill(3) for i in range(261)])
#     mask = np.array([os.path.exists(f'{indir}/{x}_tmp') for x in batches])
#     final_batches = batches[~mask]

    with ProcessPoolExecutor() as executor:
        results = list(tqdm(executor.map(load_mcds, final_batches), total=len(final_batches)))

    mcds_merged = MCDS(xr.concat(results, dim=obs_dim))
    mcds_merged.obs_dim = obs_dim
    mcds_merged.var_dim = var_dim

    mcds_merged.add_feature_cov_mean(var_dim=var_dim)

    mcds_merged.coords[f'{var_dim}_cov_mean'].values.shape, \
    mcds_merged.coords[f'{var_dim}_cov_mean'].values.mean()

    mcds_merged.add_mc_frac(
        normalize_per_cell=True,  # after calculating mC frac, per cell normalize the matrix
            clip_norm_value=10  # clip outlier values above 10 to 10
    )

    adata = mcds_merged.get_adata(mc_type=mcg_pattern,
                       var_dim=var_dim,
                       select_hvf=False)
    adata.write_h5ad(f'{projdir}/h5ad/{var_dim}/raw_norm.h5ad')

In [ ]:
!date